# 👁️ 08 — Visão AGNÓSTICA a tela (Fase 2): detector de cartas que generaliza

> **Célula única — `Runtime → Run all`.** Use **GPU** (Colab). `HF_TOKEN` nos Secrets é opcional (publica o modelo).

## O desafio
100 alunos, 100 telas (UIs) diferentes. O sistema tem que ler uma imagem 2D de **qualquer** mesa — inclusive as que **nunca viu**. Ler a carta é trivial; **generalizar a layouts novos** é o problema. A F1 (baseline por template, no backend) prova o pipeline mas **não generaliza** (cai de 100% pra ~88% em estilos não vistos). Esta é a F2: o modelo **treinado** que fecha esse gap.

## A estratégia (verificada por double-check adversarial)
- **Síntese + domain randomization no eixo ESTILO-DE-UI** é a única alavanca comprovada: gerar milhares de mesas variando fonte do rank, cor do naipe, **deck 2-cores E 4-cores**, feltro, tema, escala, posição, blur/ruído — o modelo aprende "o que é um Ás de espadas" **independente do desenho da UI**.
- **YOLO11n** (nano) só pra **localizar + classificar** as 52 cartas (a peça que precisa generalizar). Naipes desenhados como **formas** (não glifos) → renderizam sempre + variedade.
- **Prova pra banca:** avalia em **HELD-OUT de 20 estilos NUNCA vistos** (disjuntos do treino). O mAP50 nesses estilos é a evidência de que **não está enviesado**.

## Depois de rodar
Baixe o `.onnx` e plugue no backend (substitui o reconhecedor por template da F1). Fura o teto com um **fine-tune de <200 screenshots reais** de clientes de poker diversos.

*Números / OCR: para ler o pote/stacks use PaddleOCR (a F1 usa dígitos por template; o número é o elo fraco conhecido). As CARTAS são o núcleo aqui.*


In [ ]:
# ============================================================
# VISAO AGNOSTICA (Fase 2) - detector de cartas que generaliza pra QUALQUER tela 2D
# Celula UNICA: Run all. GPU (Colab). Pre-req opcional (Secrets): HF_TOKEN (publica o modelo).
#
# A ESTRATEGIA (verificada por double-check adversarial):
#  - o gargalo NAO e ler a carta, e LOCALIZAR numa UI nunca vista;
#  - a unica alavanca comprovada e SINTESE + DOMAIN RANDOMIZATION no eixo ESTILO-DE-UI;
#  - treina YOLO11n (nano) so pra LOCALIZAR+CLASSIFICAR as 52 cartas em imagem 2D chapada;
#  - mede em HELD-OUT de estilos NUNCA vistos (a prova de que generaliza, pra banca).
# ============================================================
import os, random, math, glob, time, subprocess
get_ipython().system('pip -q install ultralytics pillow numpy')

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

# ---------- logs estilo Use a Cabeca ----------
def box(t, ls=()):
    w = max([len(t)] + [len(x) for x in ls]) if ls else len(t)
    print("\n  +" + "-"*(w+2) + "+"); print("  | " + t.ljust(w) + " |"); print("  +" + "-"*(w+2) + "+")
    for x in ls: print("  | " + x.ljust(w) + " |")
    if ls: print("  +" + "-"*(w+2) + "+")

# ---------- CONFIG ----------
N_TRAIN, N_VAL = 4000, 600      # imagens sinteticas (aumente com A100)
EPOCHS, IMGSZ = 60, 640
ROOT = "/content/pokervision"
HF_MODEL = "felipesp1983/poker-arena-vision"
RANKS = "23456789TJQKA"; SUITS = "shdc"
CLASSES = [r+s for r in RANKS for s in SUITS]          # 52 classes: 'As','2h',...
RANK_DISP = {"T": "10"}

box("VISAO AGNOSTICA (F2) - o plano", [
    "1. gerar mesas 2D SINTETICAS com dezenas de ESTILOS (fonte/cor/deck/feltro/tema)",
    "2. treinar YOLO11n pra localizar+classificar as 52 cartas",
    "3. HELD-OUT: avaliar em estilos NUNCA vistos no treino -> prova de generalizacao",
])

# ---------- fontes variadas (domain randomization no estilo do rank) ----------
get_ipython().system('apt-get -qq install -y fonts-liberation fonts-dejavu-core fonts-freefont-ttf >/dev/null 2>&1')
FONTS = glob.glob("/usr/share/fonts/truetype/**/*.ttf", recursive=True)
FONTS = [f for f in FONTS if not any(b in f.lower() for b in ("emoji","symbol","mono"))][:14] or [None]
print(f"fontes p/ variar o rank: {len(FONTS)}")

# ---------- naipes desenhados (formas, nao glifos) => renderizam sempre + variedade ----------
def draw_suit(d, suit, cx, cy, s, color):
    if suit == "d":  # diamante
        d.polygon([(cx,cy-s),(cx+s*0.7,cy),(cx,cy+s),(cx-s*0.7,cy)], fill=color)
    elif suit == "h":  # coracao
        d.pieslice([cx-s,cy-s,cx,cy], 0, 360, fill=color)
        d.pieslice([cx,cy-s,cx+s,cy], 0, 360, fill=color)
        d.polygon([(cx-s*0.98,cy-s*0.1),(cx+s*0.98,cy-s*0.1),(cx,cy+s)], fill=color)
    elif suit == "s":  # espada
        d.polygon([(cx,cy-s),(cx+s*0.85,cy+s*0.4),(cx-s*0.85,cy+s*0.4)], fill=color)
        d.pieslice([cx-s*0.85,cy-s*0.1,cx,cy+s*0.6], 0, 360, fill=color)
        d.pieslice([cx,cy-s*0.1,cx+s*0.85,cy+s*0.6], 0, 360, fill=color)
        d.rectangle([cx-s*0.14,cy+s*0.2,cx+s*0.14,cy+s], fill=color)
        d.polygon([(cx-s*0.4,cy+s),(cx+s*0.4,cy+s),(cx,cy+s*0.5)], fill=color)
    else:  # club (paus)
        r = s*0.5
        d.ellipse([cx-r,cy-s*0.9,cx+r,cy+r*0.2], fill=color)
        d.ellipse([cx-s*0.9,cy-r*0.3,cx-s*0.9+2*r,cy-r*0.3+2*r], fill=color)
        d.ellipse([cx+s*0.9-2*r,cy-r*0.3,cx+s*0.9,cy-r*0.3+2*r], fill=color)
        d.rectangle([cx-s*0.14,cy,cx+s*0.14,cy+s], fill=color)
        d.polygon([(cx-s*0.4,cy+s),(cx+s*0.4,cy+s),(cx,cy+s*0.5)], fill=color)

def rand_color(rng, kind):
    j = lambda: rng.randint(-25, 25)
    base = {"black": (25,25,25), "red": (200,30,40), "blue": (30,90,210), "green": (20,120,40)}[kind]
    return tuple(int(np.clip(c + j(), 0, 255)) for c in base)

def render_card(rng, rank, suit, w):
    h = int(w*1.4); img = Image.new("RGB", (w, h), (0,0,0,0))
    bg = tuple(rng.randint(238,255) for _ in range(3))
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([0,0,w-1,h-1], radius=rng.randint(3,w//8), fill=bg,
                        outline=tuple(rng.randint(150,200) for _ in range(3)), width=rng.randint(1,2))
    four = rng.random() < 0.4  # 40% deck 4-cores
    cmap = {"s":"black","c":("green" if four else "black"),"h":"red","d":("blue" if four else "red")}
    col = rand_color(rng, cmap[suit])
    try: f = ImageFont.truetype(rng.choice(FONTS), int(w*rng.uniform(0.30,0.40)))
    except Exception: f = ImageFont.load_default()
    d.text((w*0.08, h*0.03), RANK_DISP.get(rank, rank), font=f, fill=col)
    draw_suit(d, suit, int(w*0.24), int(h*0.30), w*0.11, col)          # naipe do indice
    draw_suit(d, suit, w//2, int(h*0.62), w*rng.uniform(0.22,0.28), col)  # naipe central
    return img

def gen_sample(rng, styles_seed):
    """Uma mesa 2D + labels YOLO. styles_seed fixa o 'estilo' (fundo/escala) do conjunto."""
    W, H = 900, 600
    srng = random.Random(styles_seed)
    felt = tuple(srng.randint(15,110) for _ in range(3))
    img = Image.new("RGB", (W, H), felt)
    d = ImageDraw.Draw(img)
    if srng.random() < 0.7:  # vinheta/tema
        d.ellipse([W*0.08,H*0.03,W*0.92,H*0.97], fill=tuple(min(255,c+srng.randint(8,30)) for c in felt))
    deck = CLASSES[:]; rng.shuffle(deck)
    n_board = rng.choice([0,3,3,4,5]); hole = deck[:2]; board = deck[2:2+n_board]
    cw = int(srng.uniform(40,84))  # baralhos de TAMANHOS diferentes por aluno; labels = []
    def place(card, cx, cy):
        ci = render_card(rng, card[0], card[1], cw)
        img.paste(ci, (cx-ci.width//2, cy-ci.height//2), ci.convert("RGBA"))
        labels.append((CLASSES.index(card), cx/W, cy/H, ci.width/W, ci.height/H))
    if board:
        gap = int(cw*1.3); x0 = W//2 - (len(board)-1)*gap//2; by = H//2 + rng.randint(-25,10)
        for i,c in enumerate(board): place(c, x0+i*gap+rng.randint(-4,4), by)
    hx = W//2 + rng.randint(-50,50); hy = int(H*0.82)+rng.randint(-15,15)
    for i,c in enumerate(hole): place(c, hx-46+i*92, hy)
    if rng.random() < 0.5:  # ruido/blur leve as vezes
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0,0.8)))
    arr = np.asarray(img).astype("int16") + np.random.default_rng(styles_seed+rng.randint(0,9999)).normal(0, rng.uniform(0,6), (H,W,3))
    return Image.fromarray(np.clip(arr,0,255).astype("uint8")), labels

# ---------- gera o dataset (train com estilos 0..79 ; val = HELD-OUT 80..99) ----------
box("[1/3] Gerando dataset sintetico (estilos de TREINO x HELD-OUT disjuntos)")
for split in ("train","val"):
    os.makedirs(f"{ROOT}/images/{split}", exist_ok=True); os.makedirs(f"{ROOT}/labels/{split}", exist_ok=True)
def build(split, n, style_ids):
    rng = random.Random(hash(split) & 0xffff)
    for i in range(n):
        img, labels = gen_sample(rng, rng.choice(style_ids))
        img.save(f"{ROOT}/images/{split}/{i:05}.jpg", quality=88)
        with open(f"{ROOT}/labels/{split}/{i:05}.txt","w") as fh:
            fh.write("\n".join(f"{c} {x:.5f} {y:.5f} {w:.5f} {h:.5f}" for c,x,y,w,h in labels))
        if i % 1000 == 0: print(f"  {split}: {i}/{n}")
build("train", N_TRAIN, list(range(0,80)))       # 80 estilos de treino
build("val",   N_VAL,   list(range(80,100)))     # 20 estilos NUNCA vistos (held-out)
with open(f"{ROOT}/data.yaml","w") as fh:
    fh.write(f"path: {ROOT}\ntrain: images/train\nval: images/val\nnc: 52\nnames: {CLASSES}\n")
print("dataset pronto.")

# ---------- treina YOLO11n ----------
box("[2/3] Treinando YOLO11n (localizar+classificar 52 cartas)")
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data=f"{ROOT}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=32,
            patience=15, device=0, project=ROOT, name="cards", verbose=True)

# ---------- avalia no HELD-OUT + exporta ----------
box("[3/3] HELD-OUT (estilos nunca vistos) + export ONNX")
m = model.val(data=f"{ROOT}/data.yaml", split="val")
mAP50 = float(m.box.map50); mAP = float(m.box.map)
onnx = model.export(format="onnx", imgsz=IMGSZ, opset=12)
box("RESULTADO (a prova pra banca)", [
    f"mAP50 em ESTILOS NUNCA VISTOS: {mAP50*100:.1f}%   (localizacao+classe corretas)",
    f"mAP50-95: {mAP*100:.1f}%",
    "modelo agnostico a tela treinado so em sintetico -> generaliza pra UIs nao vistas.",
    "Proximo (fura o teto): fine-tune com <200 screenshots reais de clientes diversos.",
])
print("ONNX:", onnx)

# ---------- publica no HF (opcional) ----------
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        from huggingface_hub import HfApi
        api = HfApi(token=tok); api.create_repo(HF_MODEL, repo_type="model", exist_ok=True, private=True)
        api.upload_file(path_or_fileobj=str(onnx), path_in_repo="cards_yolo11n.onnx", repo_id=HF_MODEL, repo_type="model")
        print("publicado em", HF_MODEL)
except Exception as e:
    print("(HF opcional, pulado):", e)
print("\nFIM. Baixe o .onnx e plugue no backend (substitui o reconhecedor por template da F1).")
